# Дополнения к лабораторной 6 (вставить в Kaggle после основных ячеек)

**Порядок:** выполняй блоки там, где указано «после …». Переменные: `model`, `processor`, `get_image_features`, `get_text_features`, `class_names`, `test_dataset`, `test_loader`, при необходимости `all_preds`, `all_labels`, `all_image_features`, `all_labels_list`, `search_by_text`.

## 1. Версии (сразу после импортов / загрузки CLIP)
Воспроизводимость и проверка у преподавателя.

In [ ]:
import sys
import torch
import torchvision
import sklearn

try:
    import transformers
    tf_ver = transformers.__version__
except Exception:
    tf_ver = "n/a"

print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", tf_ver)
print("sklearn:", sklearn.__version__)
print("cuda:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

## 2. Сохранить метрики в `lab6_metrics.json` (после всех экспериментов)
Перед запуском убедись, что определены: `acc`, `prompt_results`, `best_single_t`, `worst_single_t`, `ens_acc`, `mean_recall`, `class_names`, `test_dataset`.

In [ ]:
import json

def save_lab_metrics(path="lab6_metrics.json"):
    payload = {
        "dataset": "StanfordCars",
        "n_classes": len(class_names),
        "test_size": len(test_dataset),
        "zero_shot_baseline_acc": float(acc),
        "prompts": {str(k): float(v) for k, v in prompt_results.items()},
        "best_single_template": str(best_single_t),
        "best_single_acc": float(prompt_results[best_single_t]),
        "worst_single_template": str(worst_single_t),
        "worst_single_acc": float(prompt_results[worst_single_t]),
        "ensemble_acc": float(ens_acc),
        "mean_recall_at_10": float(mean_recall),
    }
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    print("Saved:", path)
    return payload

# save_lab_metrics()

## 3. Ошибки по «бренду» (после zero-shot: `all_preds`, `all_labels`)
Считаем долю предсказаний, где **первое слово** истинного класса совпадает с первым словом предсказанного (грубый proxy «угадали марку»).

In [ ]:
def first_token(name):
    return name.split()[0] if name else ""

brand_hits = 0
for ti, pi in zip(all_labels, all_preds):
    if first_token(class_names[ti]) == first_token(class_names[pi]):
        brand_hits += 1

brand_acc = brand_hits / len(all_labels)
print(f"Грубая accuracy по первому слову (бренд): {100 * brand_acc:.2f}%")
print(f"Top-1 по полному классу: {100 * float((all_preds == all_labels).mean()):.2f}%")
print("Разница показывает долю ошибок внутри одной марки (поколение/кузов/модель).")

## 4. Precision@K для текстовых запросов (после 2A–2B)
Ручная разметка релевантности: для запроса задаёшь множество **индексов классов**, которые считаешь релевантными; считаем долю релевантных среди top-K.

In [ ]:
def precision_at_k_for_classes(query, relevant_class_indices, image_features, labels, k=6):
    """relevant_class_indices: set или list индексов класса (0..C-1)."""
    rel = set(relevant_class_indices)
    idx, _ = search_by_text(query, image_features, top_k=k)
    top_labels = labels[idx.numpy()].tolist()
    hits = sum(1 for lb in top_labels if int(lb) in rel)
    return hits / k, top_labels

# Пример: запрос про Ferrari — релевантны все классы, где в имени есть Ferrari
ferrari_idx = [i for i, n in enumerate(class_names) if "Ferrari" in n]
p, top = precision_at_k_for_classes(
    "a photo of a Ferrari car", ferrari_idx, all_image_features, all_labels_list, k=6
)
print("Ferrari query: precision@6 =", round(p, 3), "| top labels:", [class_names[j] for j in top])

porsche_idx = [i for i, n in enumerate(class_names) if "Porsche" in n]
p2, top2 = precision_at_k_for_classes(
    "a photo of a Porsche car", porsche_idx, all_image_features, all_labels_list, k=6
)
print("Porsche query: precision@6 =", round(p2, 3), "| top labels:", [class_names[j] for j in top2])

## 5. Linear probe (опционально, нужен train split)
~2–4 минуты на T4. Сравнивает zero-shot с логрегом на эмбеддингах CLIP.

In [ ]:
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader
from torchvision import datasets
import os

def find_stanford_cars_root():
    kaggle_root = "/kaggle/input"
    candidates = []
    if os.path.isdir(kaggle_root):
        for root, dirs, _ in os.walk(kaggle_root):
            if "stanford_cars" in dirs:
                candidates.append(root)
    if candidates:
        return sorted(candidates, key=len)[0]
    if os.path.isdir("./data/stanford_cars"):
        return "./data"
    raise FileNotFoundError("stanford_cars not found")

def pil_collate(batch):
    imgs, labels = zip(*batch)
    return list(imgs), torch.tensor(labels)

# раскомментируй для запуска:
# INPUT_ROOT = find_stanford_cars_root()
# train_ds = datasets.StanfordCars(root=INPUT_ROOT, split="train", download=False, transform=None)
# train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, collate_fn=pil_collate)
#
# X_train, y_train = [], []
# with torch.no_grad():
#     for images, labels in tqdm(train_loader, desc="Train features"):
#         fe = get_image_features(images).cpu().numpy()
#         X_train.append(fe)
#         y_train.extend(labels.numpy())
# X_train = np.vstack(X_train)
# y_train = np.array(y_train)
#
# X_test = all_image_features.numpy()
# y_test = all_labels_list.numpy()
#
# clf = LogisticRegression(max_iter=1000, C=0.316, random_state=42, n_jobs=-1)
# clf.fit(X_train, y_train)
# lp_acc = (clf.predict(X_test) == y_test).mean()
# print(f"Linear probe accuracy: {100 * lp_acc:.2f}%  |  Zero-shot: {100 * float(acc):.2f}%")

## 6. Вызов сохранения метрик
Раскомментируй после того как посчитан `mean_recall` из Recall@10.

In [ ]:
# save_lab_metrics("lab6_metrics.json")
# from IPython.display import FileLink
# FileLink("lab6_metrics.json")  # в Kaggle — скачать файл из output

## Шаблон выводов (Markdown в конец ноутбука)

1. **Датасет:** Stanford Cars, 196 классов, 8041 тестовых изображений.  
2. **Zero-shot:** baseline ~58.6%; лучший шаблон `a photo of a {}` ~59%; ансамбль ~59%; промпты дают ~1 п.п. разброса — CLIP чувствителен к формулировке.  
3. **Ошибки:** много путаницы внутри одной марки (fine-grained); грубая метрика «по первому слову» обычно заметно выше top-1 по полному имени класса.  
4. **Поиск:** по точным классам запрос хорошо тянет релевантные авто; абстрактные запросы дают промахи (см. precision@K).  
5. **Recall@10** по имени класса высокий — метрика мягче топ-1.  
6. **t-SNE:** 2D не обязан совмещать звёзды текста и облако изображений; важнее косинус в 512-D.  
7. **Ограничения CLIP:** английские промпты; fine-grained; абстрактные концепты без явного визуального якоря.